In [1]:
import requests                      # To download webpage HTML
from bs4 import BeautifulSoup        # To parse HTML and extract text
import pandas as pd                  # To save data to CSV
import spacy                        # NLP model for keyword extraction
import nltk                        # For stopwords
from nltk.corpus import stopwords

# Download necessary NLTK data (run only once)
nltk.download("stopwords")
nltk.download("punkt")

# Define English stopwords (common words to ignore)
stop_words = set(stopwords.words("english"))

# Load spaCy English language model
nlp = spacy.load("en_core_web_sm")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\marut\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\marut\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [2]:
pip install python-docx

Note: you may need to restart the kernel to use updated packages.


In [9]:
pip install PyMuPDF

   ---------------------------------------- 0.0/18.5 MB ? eta -:--:--
   ------ --------------------------------- 3.1/18.5 MB 15.4 MB/s eta 0:00:01
   ------------- -------------------------- 6.3/18.5 MB 15.4 MB/s eta 0:00:01
   ------------------- -------------------- 9.2/18.5 MB 15.4 MB/s eta 0:00:01
   ------------------------ --------------- 11.3/18.5 MB 13.8 MB/s eta 0:00:01
   ------------------------------ --------- 14.2/18.5 MB 14.1 MB/s eta 0:00:01
   ------------------------------------ --- 16.8/18.5 MB 13.9 MB/s eta 0:00:01
   ---------------------------------------- 18.5/18.5 MB 13.8 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [11]:
import os
import re
import csv
import docx
import fitz  # PyMuPDF

def extract_text_from_pdf(pdf_path):
    text = ""
    with fitz.open(pdf_path) as doc:
        for page in doc:
            text += page.get_text()
    return text
def extract_name(text):
    lines = text.splitlines()
    candidate_lines = [line.strip() for line in lines if line.strip()]
    candidate_lines = candidate_lines[:7]

    print("\n🔍 Checking top lines for name:")
    for line in candidate_lines:
        print(f"- {line}")
        if re.match(r"^([A-Z][a-z]+ [A-Z][a-z]+|[A-Z]+ [A-Z]+)$", line):
            print(f"✅ Name detected: {line}")
            return line

    print("⚠️ No valid name found. Returning 'N/A'")
    return "N/A"

def extract_resume_info(path):
    ext = os.path.splitext(path)[-1].lower()

    if ext == ".docx":
        doc = docx.Document(path)
        text = "\n".join([para.text for para in doc.paragraphs])
    elif ext == ".pdf":
        text = extract_text_from_pdf(path)
    else:
        print(f"Unsupported file type: {ext}")
        return {"Name": "N/A", "Email": "N/A", "Phone": "N/A", "Education": []}

    name = extract_name(text)

    # Extract email
    email_match = re.search(r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+", text)
    email = email_match.group(0) if email_match else "N/A"

    # Extract phone number
    phone_match = re.search(r"\b\d{3}[-.\s]?\d{3}[-.\s]?\d{4}\b", text)
    phone = phone_match.group(0) if phone_match else "N/A"

    # Extract education info
    educations = extract_education(text)

    return {
        "Name": name,
        "Email": email,
        "Phone": phone,
        "Education": educations
    }
def extract_education(text):
    # Find the EDUCATION section lines
    lines = text.splitlines()
    edu_lines = []
    capture = False
    for line in lines:
        if "education" in line.lower():
            capture = True
            continue
        if capture:
            # stop capture at next all uppercase section or blank line
            if line.strip() == "" or (line.isupper() and "education" not in line.lower()):
                break
            edu_lines.append(line.strip())

    # Join lines and split by blank lines to get blocks
    edu_text = "\n".join(edu_lines)
    blocks = [b.strip() for b in edu_text.split("\n\n") if b.strip()]

    for block in blocks:
        # Search for degree in block
        degree_match = re.search(r"(Bachelor|Master|Certificate|Diploma|Ph\.?D\.?|Master's Degree)", block, re.IGNORECASE)
        if degree_match:
            degree = degree_match.group()
            # Try to find year
            year_match = re.search(r"\b(19|20)\d{2}\b", block)
            year = int(year_match.group()) if year_match else None
            # Use first line as course
            course = block.splitlines()[0]
            return [{
                "level": degree,
                "course": course,
                "year": year or 0
            }]
    return []
def save_to_csv(resume_info, csv_path="candidates_info.csv"):
    file_exists = os.path.isfile(csv_path)

    with open(csv_path, mode="a", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        if not file_exists:
            writer.writerow(["Name", "Email", "Phone", "Education_Level", "Education_Course"])
        if resume_info["Education"]:
            for edu in resume_info["Education"]:
                writer.writerow([resume_info["Name"], resume_info["Email"], resume_info["Phone"], edu["level"], edu["course"]])
        else:
            writer.writerow([resume_info["Name"], resume_info["Email"], resume_info["Phone"], "N/A", "N/A"])

def process_resume(path):
    info = extract_resume_info(path)
    save_to_csv(info)
    print(f"\n✅ Info from {os.path.basename(path)} saved.")
    print(f"Name: {info['Name']}")
    print(f"Email: {info['Email']}")
    print(f"Phone: {info['Phone']}")
    for edu in info["Education"]:
        print(f"Education: {edu['level']} in {edu['course']}")



In [19]:
resume_path = r"C:\Users\marut\Desktop\resume\Maruta_Zalane_resume.pdf"
process_resume(resume_path)


🔍 Checking top lines for name:
- MARUTA ZALANE
✅ Name detected: MARUTA ZALANE

✅ Info from Maruta_Zalane_resume.pdf saved.
Name: MARUTA ZALANE
Email: maruta.zalane@gmail.com
Phone: 343-598-0069
Education: Master in Business Intelligence Systems Infrastructure
